# 01 · Tensors and a discharge hydrograph

**Question:** How much water passed the upstream gauge above a chosen discharge threshold during the days we observed?

We'll work with six made-up daily mean readings from two gauges. By the end, you should be able to select a gauge, read a hydrograph, handle a missing day, and explain how a rate becomes a volume. Run each cell in order and pause to predict the result whenever you see **Predict**.

## 1 · Read a tensor by its axes

A tensor stores values of one data type in a grid. The **shape** tells you how many entries fit along each axis. Here axis 0 is time (six days) and axis 1 is gauge (upstream and downstream). So the shape is `(6, 2)`. Each decimal value is a daily mean discharge in cubic metres per second (m³/s).

`dtype` is the data type of each entry. We use `torch.float32` for decimal readings and to represent `NaN` (a missing measurement in this example). `tuple(q.shape)` displays the axis lengths as an ordinary Python tuple.

**Predict:** Where in the tensor is the downstream reading for day 3? PyTorch starts counting at zero.

In [1]:
import torch

q = torch.tensor([
    [2.0, 4.0],                # Day 1
    [4.0, 6.0],                # Day 2
    [8.0, 10.0],               # Day 3
    [float("nan"), 11.0],       # Day 4
    [6.0, 8.0],                # Day 5
    [3.0, 5.0],                # Day 6
], dtype=torch.float32)

print("Shape and dtype", tuple(q.shape), q.dtype)
print("Downstream on day 3", q[2, 1])  # Two integers select one value, shape ().

Shape and dtype (6, 2) torch.float32
Downstream on day 3 tensor(10.)


## 2 · Select a gauge

In `q[:, 0]`, the colon keeps all days and the integer selects the upstream gauge, so the gauge axis disappears. In `q[:, :1]`, the slice selects the same gauge and preserves an axis of length one. `reshape(6, 1)` gives the one-dimensional series two axes again. It changes the arrangement of the six values without changing their order.

**Predict:** Which two expressions will give the same shape?

In [2]:
upstream = q[:, 0]                      # (6, 2) → (6,)
upstream_column = q[:, :1]               # (6, 2) → (6, 1)
upstream_reshaped = upstream.reshape(6, 1)  # (6,) → (6, 1)

print("Integer", tuple(upstream.shape))
print("Slice", tuple(upstream_column.shape))
print("Reshape", tuple(upstream_reshaped.shape))

Integer (6,)
Slice (6, 1)
Reshape (6, 1)


## 3 · Read the hydrograph and handle missing data

A hydrograph shows discharge over time. In this text version, time runs from top to bottom and each block represents 1 m³/s. Only the recorded days are shown with bars. Day 4 is unknown, so we cannot tell whether day 3 was the peak of the whole period.

`torch.isfinite` produces one Boolean per reading. Applying that mask as an index keeps only the recorded values, reducing `(6,)` to `(5,)`. Keep the original series for plotting or working with time windows. Filtering compresses the series and would make days 3 and 5 adjacent in the result.

**Predict:** What does day 4 contribute to the mean of the recorded days?

In [3]:
observed = torch.isfinite(upstream)       # (6,) → (6,) of True/False
recorded = upstream[observed]              # (6,) → (5,)

# The Python loop formats the chart. Each iteration uses one scalar reading.
for day, (value, valid) in enumerate(zip(upstream, observed), start=1):
    bar = "█" * int(value) if valid else "missing"
    print(f"Day {day}  {bar}")

print("Mask", tuple(observed.shape), observed)
print("Observed-day mean (m³/s)", recorded.mean())  # (5,) → ()

Day 1  ██
Day 2  ████
Day 3  ████████
Day 4  missing
Day 5  ██████
Day 6  ███
Mask (6,) tensor([ True,  True,  True, False,  True,  True])
Observed-day mean (m³/s) tensor(4.6000)


## 4 · Turn a flow rate into a volume

Suppose the threshold is 5 m³/s. Day 3 averaged 8 m³/s, so the excess is 3 m³/s. Over 86,400 seconds, that gives an estimated excess volume of **259,200 m³** for that day. This calculation applies the threshold to the daily mean. It cannot resolve within-day fluctuations or quantify how much water could actually be diverted.

`recorded - threshold` uses **broadcasting**. PyTorch applies the same scalar to every value without changing the shape. `clamp_min(0)` changes negative excesses to zero. Multiplying by seconds per day turns each rate into a volume. Finally, `sum()` reduces the vector to one scalar for the observed days.

**Predict:** Which other day contributes, and what is the five-day observed total?

In [4]:
threshold = 5.0  # m³/s
seconds_per_day = 24 * 60 * 60

excess_rate = (recorded - threshold).clamp_min(0)  # (5,) → (5,)
volume_per_observed_day = excess_rate * seconds_per_day  # (5,) → (5,)
observed_volume = volume_per_observed_day.sum()  # (5,) → ()

print("Excess rate (m³/s)", tuple(excess_rate.shape), excess_rate)
print("Daily excess volumes (m³)", tuple(volume_per_observed_day.shape), volume_per_observed_day)
print("Observed total (m³)", tuple(observed_volume.shape), observed_volume)
assert observed_volume.item() == 345_600  # item() extracts the scalar as a Python number.

Excess rate (m³/s) (5,) tensor([0., 0., 3., 1., 0.])
Daily excess volumes (m³) (5,) tensor([     0.,      0., 259200.,  86400.,      0.])
Observed total (m³) () tensor(345600.)


## 5 · Try it yourself

1. Select the downstream gauge twice, once with an integer and once with a slice. Predict both shapes. Check that the first and last readings match in both selections.
2. Compute downstream volume above 5 m³/s. Estimate it by hand first, then use the same PyTorch operations. Check that the tensor of excess rates has six entries and that day 4 contributes 518,400 m³.
3. Compare both gauges over the **same observed days**. Build a mask that is true when both readings are finite. Use a threshold tensor with one value per gauge, `[5.0, 7.0]`. Predict the resulting daily volume tensor's shape, then reduce it to one total per gauge. Explain which axis disappeared.

<details>
<summary>Hint for exercise 3</summary>

Start with `torch.isfinite(q).all(dim=1)`. The `all` asks whether every gauge is valid on each day. Use that mask to keep whole rows, then subtract the two thresholds. To get one total per gauge, sum over time.

</details>

In [7]:
d_int = q[:,1] # (6, 2) -> (6,)
d_slice = q[:, 1:] # (6, 2) -> (6, 1)
print("Integer", tuple(d_int.shape))
print("Slice", tuple(d_slice.shape))

Integer (6,)
Slice (6, 1)


In [8]:
excess_d = (d_int - threshold).clamp_min(0)  # (6,) → (6,)
volume_per_day_d = excess_d * seconds_per_day  # (6,) → (6,)
observed_volume_d = volume_per_day_d.sum()  # (6,) → ()
print("Excess rate (m³/s)", tuple(excess_d.shape), excess_d)
print("Daily excess volumes (m³)", tuple(volume_per_day_d.shape), volume_per_day_d)
print("Observed total (m³)", tuple(observed_volume_d.shape), observed_volume_d)

Excess rate (m³/s) (6,) tensor([0., 1., 5., 6., 3., 0.])
Daily excess volumes (m³) (6,) tensor([     0.,  86400., 432000., 518400., 259200.,      0.])
Observed total (m³) () tensor(1296000.)


In [9]:
observed = torch.isfinite(q).all(dim=1)  # (6, 2) → (6, 2) of True/False
recorded = q[observed]              # (6, 2) → (5, 2)

threshold = torch.tensor([[5.0, 7.0]])  # (1, 2)

excess = (recorded - threshold).clamp_min(0)  # (5, 2) → (5, 2)
volume_per_observed_day = excess * seconds_per_day  # (5, 2) → (5, 2)
observed_volume = volume_per_observed_day.sum()  # ()

print("Excess rate (m³/s)", tuple(excess.shape), excess)
print("Daily excess volumes (m³)", tuple(volume_per_observed_day.shape), volume_per_observed_day)
print("Observed total (m³)", tuple(observed_volume.shape), observed_volume)

Excess rate (m³/s) (5, 2) tensor([[0., 0.],
        [0., 0.],
        [3., 3.],
        [1., 1.],
        [0., 0.]])
Daily excess volumes (m³) (5, 2) tensor([[     0.,      0.],
        [     0.,      0.],
        [259200., 259200.],
        [ 86400.,  86400.],
        [     0.,      0.]])
Observed total (m³) () tensor(691200.)


## Before lesson 2

For extra practice, run `python 01_tensor_shapes.py` from the repository root. It gives you a smaller tensor for experimenting with reductions and broadcasting.

In your own words, explain why `(6,)` and `(6, 1)` describe different shapes, why a missing day is different from a zero-flow day, and why the volume result depends on the time resolution. Bring whichever part took the most effort to the next discussion so we can adjust the pace.